In [0]:
dbutils.widgets.text("catalog", "vp_ce_2026_07_1382", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
-- VP Q01 Daily — energy metrics per day, ascending
SELECT
  session_date,
  SUM(completed_sessions) AS completed_sessions,
  SUM(energy_kwh)         AS energy_kwh,
  SUM(billed_amount)     AS total_billed,
  SUM(amount_paid)       AS total_paid
FROM IDENTIFIER(:catalog || ".gold.agg_daily_station")
GROUP BY session_date
ORDER BY session_date ASC

In [0]:
%sql
-- VP Q02 Station — utilisation per station, descending
-- utilisation_pct = 100 × COMPLETED minutes ÷ (station's chargers × 1,440 × days)
SELECT
  station_id,
  city,
  SUM(completed_sessions) AS completed_sessions,
  SUM(energy_kwh)        AS energy_kwh,
  AVG(utilisation_pct)   AS utilisation_pct
FROM IDENTIFIER(:catalog || ".gold.agg_daily_station")
GROUP BY station_id, city
ORDER BY utilisation_pct DESC

In [0]:
%sql
    
-- VP Q03 Billing by plan — metrics per plan type
SELECT
  plan_at_session,
  COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END)          AS completed_sessions,
  COUNT(CASE WHEN billing_status = 'MISMATCH' THEN 1 END)   AS mismatch_sessions,
  SUM(CASE WHEN billing_status = 'MISMATCH'
           THEN ABS(billed_amount - amount_paid) ELSE 0 END)  AS revenue_leakage
FROM IDENTIFIER(:catalog || ".gold.fact_sessions")
GROUP BY plan_at_session
ORDER BY CASE plan_at_session
  WHEN 'BASIC' THEN 1
  WHEN 'PRO'   THEN 2
  WHEN 'FLEET' THEN 3
  WHEN 'GUEST' THEN 4
  ELSE 5
END

In [0]:
%sql
    
-- VP Q04 Charger — chargers with highest failure rates
-- failure_rate_pct = 100 × failed ÷ (completed + failed)
-- Only chargers with ≥ 10 completed + failed; top 10 by failure rate, then charger_id
SELECT
  charger_id,
  COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END)  AS completed_sessions,
  COUNT(CASE WHEN status != 'COMPLETED' THEN 1 END)   AS failed_sessions,
  100 * COUNT(CASE WHEN status != 'COMPLETED' THEN 1 END)
       / (COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END) + COUNT(CASE WHEN status != 'COMPLETED' THEN 1 END)) AS failure_rate_pct
FROM IDENTIFIER(:catalog || ".gold.fact_sessions")
GROUP BY charger_id
HAVING COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END) + COUNT(CASE WHEN status != 'COMPLETED' THEN 1 END) >= 10
ORDER BY failure_rate_pct DESC, charger_id
LIMIT 10

In [0]:
# VP Q05 City scorecard — single-city metrics for a date range (inclusive)
dbutils.widgets.text("city", "", "City")
dbutils.widgets.text("start_date", "2024-01-01", "Start Date (inclusive)")
dbutils.widgets.text("end_date", "2024-12-31", "End Date (inclusive)")

city       = dbutils.widgets.get("city")
start_date = dbutils.widgets.get("start_date")
end_date   = dbutils.widgets.get("end_date")

q = f"""
SELECT
  city,
  SUM(completed_sessions) AS completed_sessions,
  SUM(energy_kwh)         AS energy_kwh,
  SUM(billed_amount)     AS total_billed,
  100 * SUM(completed_sessions)
       / NULLIF(SUM(completed_sessions) + SUM(failed_sessions), 0) AS success_rate_pct
FROM {catalog}.gold.agg_daily_station
WHERE city = '{city}'
  AND session_date >= '{start_date}'
  AND session_date <= '{end_date}'
GROUP BY city
"""
display(spark.sql(q))